In [4]:
w = 78.31
t = 5.08 
c = 0.4 * w 
P_max = 8000
P_min = 800 
kt = 2.62

S_nom = 2 * P_max *  (2*w + c) / (t * (w-c)**2)
S_max = kt * S_nom
print("Stress Nominal:", S_nom )
print("Stress Actual:", S_max)


S_m = (S_max + S_min)/2
S_a = (S_max - S_min)/2
print("Stress m: ", S_mean)
print("Stress a: ", S_alt)
S_f = 1758
S_equiv = S_alt / (1 + (S_mean/S_f))
print("Stress Equiv: ", S_equiv)
b = -0.0977
N = 0.5 * (S_equiv / S_f) ** (1/b)
print("Number of Cycles: ", N)
S_y = 753
S_fl = 1
FOS = 1/((S_mean/S_y)  + (S_alt/S_f))
print("Factor of Safety: ", FOS)

Stress Nominal: 268.13146888116444
Stress Actual: 702.5044484686508


NameError: name 'S_min' is not defined

In [14]:
import numpy as np
import Scripts.Finite_Elements.FEA_3D as FEA_3D
import scipy as scipy
B1 = 5.08
B2 = 5.08
H1 = 0.6 * 78.31
H2 = 0.6 * 78.31 - 5.08
I1 = B1 * (H1 ** 3) / 12
I2 = B2 * (H2 ** 3) / 12
A1 = B1 * H1 
A2 = B2 * H2
E = 206900
L1 = 0.355 * 78.31
L2 = 0.7 * 78.31
k1 = (E * I1 / L1 ** 3) * np.array([[L1**2 * A1/I1, 0, 0, -L1**2 * A1/I1, 0, 0],
                                    [0, 12, 6*L1, 0, -12, 6*L1],
                                    [0, 6*L1, 4*L1**2, 0,-6*L1, 2*L1**2],
                                    [-L1**2 * A1/I1, 0, 0, L1**2 * A1/I1, 0, 0],
                                    [0, -12, -6*L1, 0, 12, -6*L1],
                                    [0, 6*L1, 2*L1**2, 0, -6*L1, 4*L1**2]])
k2 = (E * I2 / L2 ** 3) * np.array([[L2**2 * A2/I2, 0, 0, -L2**2 * A2/I2, 0, 0],
                                    [0, 12, 6*L2, 0, -12, 6*L2],
                                    [0, 6*L2, 4*L2**2, 0,-6*L2, 2*L2**2],
                                    [-L2**2 * A2/I2, 0, 0, L2**2 * A2/I2, 0, 0],
                                    [0, -12, -6*L2, 0, 12, -6*L2],
                                    [0, 6*L2, 2*L2**2, 0, -6*L2, 4*L2**2]])
k1_transform = np.array([[0,-1,0,0,0,0],[1,0,0,0,0,0],[0,0,1,0,0,0],[0,0,0,0,0,-1],[0,0,0,0,1,0],[0,0,0,0,0,1]]) #90 deg
k2_transform = np.array([[-1,0,0,0,0,0],[0,-1,0,0,0,0],[0,0,1,0,0,0],[0,0,0,0,-1,0],[0,0,0,0,0,-1],[0,0,0,0,0,1]]) # 180 deg
k1_g = np.transpose(k1_transform) @ k1 @ k1_transform
k2_g = np.transpose(k2_transform) @ k2 @ k2_transform
kg = np.zeros([9,9])
kg[0:6,0:6] = kg[0:6,0:6] + k1_g
kg[3:9,3:9] = kg[3:9,3:9] + k2_g
up_index = np.array([2,3,4,5,6,7,8,0,1])
kg_ri = FEA_3D.rearrange_stiffness_matrix(kg, up_index)
kuu, kup, kpu, kpp = FEA_3D.partition_stiffness_matrix(kg_ri, 7)
fu = np.array([0,0,0,0,0,8000,0])
du = scipy.linalg.lstsq(kuu, fu)[0]

print("Displacement (mm):", du[5])

Displacement (mm): 0.002489114346967778
